In [1]:
import folium
from folium.plugins import HeatMap
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

In [2]:
df = pd.read_csv("southern_california_cleaned_declustered.csv")

geometry = [Point(xy) for xy in zip(df["longitude"], df["latitude"])]
gdf = gpd.GeoDataFrame(df, crs="EPSG:4326", geometry=geometry)

def get_style(mag):
    if mag >= 6.0:
        return {"color": "#800080", "radius": 11, "weight": 2.5}  # Purple
    elif mag >= 5.0:
        return {"color": "#DC143C", "radius": 8, "weight": 1.5}  # Red
    elif mag >= 4.0:
        return {"color": "#FF8C00", "radius": 5, "weight": 1.0}  # Orange
    return {"color": "#2E8B57", "radius": 3, "weight": 0.5}  # Green


# 2. Initialize the Canvas
center_lat, center_lon = gdf["latitude"].mean(), gdf["longitude"].mean()
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=7,
    tiles="CartoDB positron",
    prefer_canvas=True,
    width="100%",
    height="650px",
)

# LAYER 1: Heatmap
heat_data = [
    [r["latitude"], r["longitude"], r["magnitude"]] for _, r in df.iterrows()
]
HeatMap(
    heat_data,
    name="Layer: Seismic Density Glow",
    radius=14,
    blur=10,
    gradient={0.2: "blue", 0.4: "lime", 0.7: "yellow", 1.0: "red"},
).add_to(m)

# LAYER 2: Significant Quakes (3.5 <= M < 5.5)
sig_quakes = df[(df["magnitude"] >= 3.5) & (df["is_mainshock"] == False)]
q_group = folium.FeatureGroup(name="Layer: Significant Quakes (M ≥ 3.5)")

for _, row in sig_quakes.iterrows():
    style = get_style(row["magnitude"])
    place_clean = str(row["place"]).split(",")[0]
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=style["radius"],
        color=style["color"],
        fill=True,
        fill_color=style["color"],
        fill_opacity=0.7,
        weight=style["weight"],
        tooltip=f"<b>M {row['magnitude']}</b> &bull; {place_clean}",
    ).add_to(q_group)

q_group.add_to(m)

# LAYER 3: Major Mainshocks (VIP Popups)
ms_group = folium.FeatureGroup(name="Layer: Major Mainshocks (M ≥ 5.5)")

for _, row in df[df["is_mainshock"] == True].iterrows():
    card = f"<b>M {row['magnitude']} Mainshock</b><br>{row['place']}<hr>Depth: {row['depth_km']} km<br>Date: {str(row['timestamp'])[:10]}"
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=13,
        color="black",
        fill=True,
        fill_color="#800080",
        fill_opacity=0.95,
        weight=3.5,
        tooltip=f"⭐ <b>MAINSHOCK: M {row['magnitude']}</b>",
        popup=folium.Popup(card, max_width=250),
    ).add_to(ms_group)

ms_group.add_to(m)

# Add Layer controller
folium.LayerControl(collapsed=False).add_to(m)

# IN JUPYTER, THIS SINGLE VARIABLE ON THE LAST LINE RENDERS THE MAP:
m